In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def preprocess_abalone_data(
    url: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Loads the Abalone dataset, encodes features, and splits into train/test."""
    # Load dataset
    columns = [
        "Sex",
        "Length",
        "Diameter",
        "Height",
        "Whole_weight",
        "Shucked_weight",
        "Viscera_weight",
        "Shell_weight",
        "Rings",
    ]
    df = pd.read_csv(url, header=None, names=columns)

    # Biological target conversion: Age = Rings + 1.5
    df["Age"] = df["Rings"] + 1.5
    df.drop(columns=["Rings"], inplace=True)

    # One-Hot Encoding for categorical "Sex" (M, F, I)
    df = pd.get_dummies(df, columns=["Sex"], drop_first=True)

    X = df.drop(columns=["Age"])
    y = df["Age"]

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Standardize scale for continuous attributes
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=X_train.columns
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test), columns=X_test.columns
    )

    return X_train_scaled, X_test_scaled, y_train, y_test


# Usage
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
X_train, X_test, y_train, y_test = preprocess_abalone_data(url)


In [2]:
import numpy as np
from sklearn.metrics import root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor


def train_decision_tree(X_train, X_test, y_train, y_test):
    """Trains and tunes a Decision Tree Regressor."""
    # Constraining depth prevents excessive branching and overfitting
    dt_model = DecisionTreeRegressor(
        max_depth=5, min_samples_split=10, random_state=42
    )
    dt_model.fit(X_train, y_train)

    train_pred = dt_model.predict(X_train)
    test_pred = dt_model.predict(X_test)

    print(
        f"DT Train RMSE: {root_mean_squared_error(y_train, train_pred):.3f}"
    )
    print(f"DT Test RMSE: {root_mean_squared_error(y_test, test_pred):.3f}")
    return dt_model


dt_model = train_decision_tree(X_train, X_test, y_train, y_test)


DT Train RMSE: 2.150
DT Test RMSE: 2.316


In [3]:
from sklearn.ensemble import RandomForestRegressor


def train_random_forest(X_train, X_test, y_train, y_test):
    """Trains and tunes a Random Forest Regressor."""
    # Averaging predictions reduces variance, depth limits combat individual overfitting
    rf_model = RandomForestRegressor(
        n_estimators=100, max_depth=8, max_features="sqrt", random_state=42
    )
    rf_model.fit(X_train, y_train)

    train_pred = rf_model.predict(X_train)
    test_pred = rf_model.predict(X_test)

    print(
        f"RF Train RMSE: {root_mean_squared_error(y_train, train_pred):.3f}"
    )
    print(f"RF Test RMSE: {root_mean_squared_error(y_test, test_pred):.3f}")
    return rf_model


rf_model = train_random_forest(X_train, X_test, y_train, y_test)


RF Train RMSE: 1.714
RF Test RMSE: 2.218


In [6]:
from xgboost import XGBRegressor


def train_xgboost(X_train, X_test, y_train, y_test):
    """Trains and tunes an XGBoost Regressor."""
    # learning_rate dictates step adjustments; subsample reduces row overfitting
    xgb_model = XGBRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42,
    )
    xgb_model.fit(X_train, y_train)

    train_pred = xgb_model.predict(X_train)
    test_pred = xgb_model.predict(X_test)

    print(
        f"XGB Train RMSE: {root_mean_squared_error(y_train, train_pred):.3f}"
    )
    print(f"XGB Test RMSE: {root_mean_squared_error(y_test, test_pred):.3f}")
    return xgb_model


xgb_model = train_xgboost(X_train, X_test, y_train, y_test)


ModuleNotFoundError: No module named 'xgboost'